# Redrob preprocess — Colab GPU + Gemini

- **CUDA GPU** → fast BGE embeddings (~15–45 min for 100K)
- **Gemini API** → JD parse + archetype labels (optional, uses your API key)
- Outputs → **`artifacts/`** (download zip and copy to your local repo)

**Before running:** Runtime → Change runtime type → **T4 GPU**

**Gemini key (recommended):** Colab **Secrets** (key icon) → add `GEMINI_API_KEY` → enable notebook access

In [ ]:
# Mount Google Drive (optional — store candidates.jsonl / repo here)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# --- CONFIG: project path (pick one option) ---
import os
from pathlib import Path

# Option A: clone from GitHub (recommended)
REPO_URL = 'https://github.com/ashokbugude/recruiter-candidate.git'  # <-- edit
if not Path('/content/recruiter-candidate/rank.py').exists():
    !git clone {REPO_URL} /content/recruiter-candidate

# Option B: uploaded zip — uncomment:
# !unzip -q /content/recruiter-candidate.zip -d /content
# Option C: Google Drive — uncomment:
# %cd /content/drive/MyDrive/recruiter-candidate

PROJECT_ROOT = Path('/content/recruiter-candidate')
assert (PROJECT_ROOT / 'rank.py').exists(), f'Project not found: {PROJECT_ROOT}'
os.chdir(PROJECT_ROOT)
print('Project root:', PROJECT_ROOT.resolve())

In [ ]:
# Verify CUDA
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Install dependencies (pinned numpy 1.26.4)
!pip install -q -r colab/requirements.txt

In [ ]:
# Upload candidates.jsonl if missing (large file ~500MB)
from pathlib import Path
candidates_path = PROJECT_ROOT / 'challenge' / 'candidates.jsonl'
if not candidates_path.exists():
    from google.colab import files
    print('Upload candidates.jsonl')
    uploaded = files.upload()
    (PROJECT_ROOT / 'challenge').mkdir(parents=True, exist_ok=True)
    for name, data in uploaded.items():
        candidates_path.write_bytes(data)
        print('Saved', candidates_path)
else:
    print('Found', candidates_path, f'({candidates_path.stat().st_size // 1_048_576} MB)')

In [ ]:
# Optional: upload partial artifacts from your PC (skip if starting fresh)
# Upload into artifacts/ — see colab/inputs/README.md
from pathlib import Path
art = PROJECT_ROOT / 'artifacts'
art.mkdir(exist_ok=True)
existing = [p.name for p in art.glob('*') if p.is_file()]
print('artifacts/ files:', existing or '(empty — will build all)')

In [ ]:
# --- Gemini + run options ---
USE_GEMINI = True   # False = silver labels only (no API spend)
FORCE = True        # True = rebuild artifacts (first Colab run)

import os
from getpass import getpass

if USE_GEMINI:
    # Prefer Colab Secrets: sidebar key icon → GEMINI_API_KEY
    try:
        from google.colab import userdata
        os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
        print('GEMINI_API_KEY loaded from Colab Secrets')
    except Exception:
        os.environ['GEMINI_API_KEY'] = getpass('Paste GEMINI_API_KEY (input hidden): ')
        print('GEMINI_API_KEY loaded from prompt')

    os.environ['REDROB_SKIP_GEMINI'] = '0'
    os.environ.setdefault('REDROB_GEMINI_FLASH_MODEL', 'gemini-3.5-flash')
    os.environ.setdefault('REDROB_GEMINI_PRO_MODEL', 'gemini-2.5-pro')
else:
    os.environ['REDROB_SKIP_GEMINI'] = '1'
    print('Gemini disabled — using silver/heuristic labels only')

In [ ]:
# Run full preprocess: Gemini (JD + labels) + CUDA embeddings → artifacts/
import sys
cmd = [sys.executable, 'colab/run_preprocess_gpu.py']
if FORCE:
    cmd.append('--force')
cmd.append('--no-skip-llm' if USE_GEMINI else '--skip-llm')
!{' '.join(cmd)}

### Embeddings only (if other artifacts already uploaded)

Uncomment and run instead of the full pipeline cell above:

In [ ]:
# !python scripts/preprocess.py --step embeddings --force --skip-llm --device cuda

In [ ]:
# Verify outputs
from pathlib import Path
import polars as pl
art = PROJECT_ROOT / 'artifacts'
checks = [
    'jd_requirements.json',
    'gemini_tiers.parquet',
    'bge_embeddings.npy',
    'faiss.index',
    'candidate_id_index.json',
    'candidate_features.parquet',
    'bm25.pkl',
]
for name in checks:
    p = art / name
    ok = p.exists()
    extra = ''
    if ok and name.endswith('.parquet'):
        extra = f' rows={pl.read_parquet(p).height}'
    elif ok:
        extra = f' size={p.stat().st_size // 1_048_576}MB'
    print(('OK  ' if ok else 'MISS'), name, extra)

if (art / 'gemini_tiers.parquet').exists():
    g = pl.read_parquet(art / 'gemini_tiers.parquet')
    if 'label_source' in g.columns:
        print('Gemini label sources:', g.group_by('label_source').len())
if (art / 'jd_requirements.json').exists():
    import json
    jd = json.loads((art / 'jd_requirements.json').read_text())
    print('JD source:', jd.get('source'))

In [ ]:
# Download artifacts zip to your PC → extract into local repo artifacts/
from google.colab import files
zip_path = PROJECT_ROOT / 'colab' / 'artifacts_download.zip'
if not zip_path.exists():
    !python colab/run_preprocess_gpu.py --no-zip --skip-llm
    import zipfile
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for p in (PROJECT_ROOT / 'artifacts').rglob('*'):
            if p.is_file() and not p.name.endswith('.log'):
                zf.write(p, arcname=str(p.relative_to(PROJECT_ROOT)))
files.download(str(zip_path))